In [9]:
import os
import pandas as pd
from dotenv import load_dotenv
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv()

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_community.vectorstores import FAISS


# ----------------------------
# CONFIG
# ----------------------------
CSV_PATH = r"C:\Users\surya.adatravu\Documents\CONV_WINDOW_LLM_ANALYSIS\RA_FSM_QA.csv"
INDEX_DIR = r"C:\Users\surya.adatravu\Documents\ContextRAG\vector_db"
SESSION_ID = "pdf_faiss_session"


# ----------------------------
# LLM + EMBEDDINGS
# ----------------------------
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


# ----------------------------
# HISTORY STORE (IN-MEMORY)
# ----------------------------
store = {}

def get_history(session_id: str) -> ChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


# ----------------------------
# LOAD HISTORY FROM CSV
# ----------------------------
def preload_history_from_csv(session_id: str, csv_path: str, max_rows=None, clear_existing=True):
    df = pd.read_csv(csv_path)
    if not {"Question", "Answer"}.issubset(df.columns):
        raise ValueError(f"CSV must contain columns Question, Answer. Found: {list(df.columns)}")

    history = get_history(session_id)
    if clear_existing:
        history.clear()

    loaded = 0
    for _, row in df.iterrows():
        if max_rows is not None and loaded >= max_rows:
            break
        history.add_user_message(str(row["Question"]))
        history.add_ai_message(str(row["Answer"]))
        loaded += 1

    print(f"✅ Loaded {loaded} Q/A pairs into history (messages={len(history.messages)}) for session '{session_id}'")
    return history


# ----------------------------
# LOAD FAISS
# ----------------------------
def load_faiss(index_dir: str) -> FAISS:
    faiss_path = os.path.join(index_dir, "index.faiss")
    pkl_path = os.path.join(index_dir, "index.pkl")
    if not (os.path.exists(faiss_path) and os.path.exists(pkl_path)):
        raise FileNotFoundError(f"FAISS index not found in {index_dir}. Run Part 1 first.")
    return FAISS.load_local(index_dir, embeddings, allow_dangerous_deserialization=True)


# ----------------------------
# QUERY REWRITE (HISTORY-AWARE)
# ----------------------------
rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Rewrite the user's latest question into a standalone search query.\n"
     "Use chat history only to resolve pronouns and references.\n"
     "Return ONLY the rewritten query."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

rewrite_chain = rewrite_prompt | llm
rewrite_with_history = RunnableWithMessageHistory(
    rewrite_chain,
    get_history,
    input_messages_key="input",
    history_messages_key="history",
)


# ----------------------------
# STRICT RAG ANSWER PROMPT (USES HISTORY + CONTEXT)
# ----------------------------
rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a strict knowledge assistant.\n"
     "Use chat history only to interpret the question.\n"
     "You must ONLY answer using the provided CONTEXT.\n"
     "If answer is not in context, say exactly:\n"
     "\"I don't know from provided knowledge.\""),
    MessagesPlaceholder(variable_name="history"),
    ("human",
     "CONTEXT:\n{context}\n\n"
     "QUESTION:\n{question}\n\n"
     "Answer using ONLY the context.")
])

rag_chain = rag_prompt | llm
rag_with_history = RunnableWithMessageHistory(
    rag_chain,
    get_history,
    input_messages_key="question",
    history_messages_key="history",
)


# ----------------------------
# OPTIONAL VALIDATION (ANSWER CLOSE TO RETRIEVED CONTEXT)
# ----------------------------
def similarity_score(text1, text2):
    v1 = embeddings.embed_query(text1)
    v2 = embeddings.embed_query(text2)
    return float(cosine_similarity([v1], [v2])[0][0])

def validate_answer_against_context(answer, retrieved_docs, threshold=0.80):
    best = -1.0
    for d in retrieved_docs:
        best = max(best, similarity_score(answer, d.page_content))
    return best >= threshold, best


# ----------------------------
# ASK FUNCTION
# ----------------------------
def ask(question, vectorstore, top_k=6, distance_threshold=0.75, validation_threshold=0.80):
    # 1) rewrite using history
    rewritten = rewrite_with_history.invoke(
        {"input": question},
        config={"configurable": {"session_id": SESSION_ID}}
    ).content.strip()

    # 2) retrieve from FAISS
    docs_with_scores = vectorstore.similarity_search_with_score(rewritten, k=top_k)
    if not docs_with_scores:
        return {"question": question, "rewritten_query": rewritten, "answer": "I don't know from provided knowledge."}

    top_distance = float(docs_with_scores[0][1])
    if top_distance > distance_threshold:
        return {"question": question, "rewritten_query": rewritten, "answer": "I don't know from provided knowledge."}

    retrieved_docs = [d for d, _ in docs_with_scores]

    context = "\n\n---\n\n".join(
        [
            f"[file={d.metadata.get('source_file','?')} page={d.metadata.get('page','?')}]\n{d.page_content}"
            for d in retrieved_docs
        ]
    )

    # 3) answer using (history + context)
    resp = rag_with_history.invoke(
        {"question": question, "context": context},
        config={"configurable": {"session_id": SESSION_ID}}
    )
    answer = resp.content.strip()
    ok, score = validate_answer_against_context(answer, retrieved_docs, threshold=validation_threshold)
    print(ok)
    if ok == False:
        print("***********************************")
        print(question)
        print(answer)
        print(score)
        print("&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&")

    # 4) validate
    # if answer != "I don't know from provided knowledge.":
    #     ok, score = validate_answer_against_context(answer, retrieved_docs, threshold=validation_threshold)
    #     if not ok:
    #         return {
    #             "question": question,
    #             "rewritten_query": rewritten,
    #             "answer": "Rejected: hallucination detected",
    #             "retrieval_distance": top_distance,
    #             "validation_score": score,
    #         }

    # return {
    #     "question": question,
    #     "rewritten_query": rewritten,
    #     "answer": answer,
    #     "retrieval_distance": top_distance,
    # }


# ----------------------------
# MAIN
# ----------------------------
if __name__ == "__main__":
    # 1) Load history into the same store used by runnables
    preload_history_from_csv(SESSION_ID, CSV_PATH, max_rows=None, clear_existing=True)

    # 2) Load FAISS index
    vs = load_faiss(INDEX_DIR)

    # 3) Inference
    print(ask("provide details of all states in finite state machine", vs))
    print(ask("provide details of confidence threshold levels", vs))
    print(ask("what about relevance threshold?", vs))


✅ Loaded 50 Q/A pairs into history (messages=100) for session 'pdf_faiss_session'
True
None
False
***********************************
provide details of confidence threshold levels
The confidence threshold levels in RA–FSM are represented as discrete values, which are:

1. **0**: Indicates no confidence in the answer; the system cannot provide a reliable response.
2. **0.25**: Suggests low confidence; the answer may be partially informed but not reliable.
3. **0.5**: Represents moderate confidence; the system has some basis for the answer, but it is not strong.
4. **0.75**: Indicates high confidence; the system is fairly certain about the answer and has substantial evidence to support it.
5. **1.0**: Represents full confidence; the system is completely certain about the answer, backed by strong evidence.

A confidence score of **0.75 or higher** is typically required for the system to proceed with providing an answer.
0.6478475994808688
&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&
None
{'quest